In [2]:
with open('input.txt', 'r') as f:
    text = f.read()
type(text)

str

In [3]:
len(text)

1115394

In [4]:
text[:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [9]:
# 1.从数据集中构造出词汇表（但由于这里是char level，所以叫字符集-chars更贴切）
chars = sorted(list(set(s for s in text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [15]:
# 2.建立stoi和itos，并基于此得到 encode、decode 函数
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for i, s in enumerate(chars)}

encode = lambda x : [stoi[s] for s in x]
decode = lambda x : ''.join([itos[i] for i in x])

print(encode("it's a good day!"))

[47, 58, 5, 57, 1, 39, 1, 45, 53, 53, 42, 1, 42, 39, 63, 2]


In [16]:
print(decode(encode("it's a good day!")))

it's a good day!


In [18]:
# 3.将数据集字符串映射成vocab-index的tensor
import torch
data = torch.tensor(encode(text))
print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [19]:
# 4.9:1 split dataset
n1 = int(0.9*len(data))
train_data = data[:n1]
val_data = data[n1:]

In [26]:
# 5.定义block_size（每次训练使用一个chunk，可参考的最大context length） = 8和 batch_size = 4（独立不重叠的chunk数量）
# 6.def get_batch(split) -> x, y
torch.manual_seed(1337)
block_size = 8
batch_size = 4

def get_batch(split : str) -> tuple[torch.tensor, torch.tensor]:
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))   # batch下每个 chunk 的 start idx
    x_batch = torch.stack([data[i:i+block_size] for i in ix], dim=0)
    y_batch = torch.stack([data[i+1:i+block_size+1] for i in ix], dim=0)
    # "hello." -> x : "hello"; y : "ello." 
    return x_batch, y_batch

xb, yb = get_batch('train')
print(xb.shape)
print(yb.shape)

print("example input:")
print(xb[0, :])
print(decode(xb[0, :].tolist()))
print("example target:")
print(yb[0, :])
print(decode(yb[0, :].tolist()))


torch.Size([4, 8])
torch.Size([4, 8])
example input:
tensor([24, 43, 58,  5, 57,  1, 46, 43])
Let's he
example target:
tensor([43, 58,  5, 57,  1, 46, 43, 39])
et's hea


#### ⭐给模型的训练任务

In [27]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context}, target is {target}")

when input is tensor([24]), target is 43
when input is tensor([24, 43]), target is 58
when input is tensor([24, 43, 58]), target is 5
when input is tensor([24, 43, 58,  5]), target is 57
when input is tensor([24, 43, 58,  5, 57]), target is 1
when input is tensor([24, 43, 58,  5, 57,  1]), target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]), target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]), target is 39
when input is tensor([44]), target is 53
when input is tensor([44, 53]), target is 56
when input is tensor([44, 53, 56]), target is 1
when input is tensor([44, 53, 56,  1]), target is 58
when input is tensor([44, 53, 56,  1, 58]), target is 46
when input is tensor([44, 53, 56,  1, 58, 46]), target is 39
when input is tensor([44, 53, 56,  1, 58, 46, 39]), target is 58
when input is tensor([44, 53, 56,  1, 58, 46, 39, 58]), target is 1
when input is tensor([52]), target is 58
when input is tensor([52, 58]), target is 1
when input is tensor([52, 58,  1]), tar

模型data flow
#### input从raw data取出，得到(B, T)的索引数据 - `ix` tensor，经过embedding层（花式索引...）转换为X
#### X的维度：(B, T, C) - batch、timestep、channels
#### target Y的维度（与input一致）：(B, T) - batch、timestep  存的是词汇表idx

In [44]:
# 7.回忆Bigram Model，仅训练一个Lookup Table，输入数据直接用上面定义的
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # lookup table - element(i, j) 表达了 序列中prev = i-char, next = j-char的共现打分
        self.token_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)

    def forward(self, ix, target):   # ix和target都是(B, T)的整数index tensor
        logits = self.token_embedding_table(ix)   # [B, T] -> [B, T, C(这里实际为vocab_size)]
        B, T, C = logits.shape
        if target is None:   # 不需要算loss
            loss = None
        else:
            logits = logits.view(B*T, C)
            targets = target.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):   # idx是表示成当前context的(B, T)整数tensor
        # idx 的 T 维度从 t=T 经generate延展成 t=T+max_new_tokens 的 new idx然后输出
        with torch.no_grad():
            for _ in range(max_new_tokens):
                logits, _ = self.forward(idx, None)
                cur_time_logits = logits[:, -1, :]
                probs = F.softmax(cur_time_logits, dim=-1)
                next_idx = torch.multinomial(probs, 1)
                idx = torch.cat((idx, next_idx), dim=1)
        return idx
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(f"loss is {loss.item()}")

torch.Size([32, 65])
loss is 4.508350849151611


In [45]:
import torch.optim as optim
model = BigramLanguageModel(vocab_size)
lr = 1e-3
batch_size = 32
optimizer = optim.AdamW(model.parameters(), lr=lr)
device = "cpu"

for i in range(10000):
    xb, yb = get_batch('train')
    _, loss = model.forward(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 1000 == 0:
        print(f"loss is {loss.item()}")

loss is 4.512293815612793
loss is 3.696620464324951
loss is 3.0240931510925293
loss is 2.727372169494629
loss is 2.6275269985198975
loss is 2.4561147689819336
loss is 2.3750240802764893
loss is 2.360565423965454
loss is 2.3664424419403076
loss is 2.4101524353027344


In [46]:
idx, _ = get_batch("val")
max_new_tokens = 100

print(f"input: {decode(idx[0, :].tolist())}")
new_idx = model.generate(idx, max_new_tokens)
print(f"after generation: {decode(new_idx[0, :].tolist())}")

input: Pedant:

after generation: Pedant:

PENThe.
f;





BRUCoubouthokes riox, n llees w'ce m geateve t an wiongh fo ou, t nowount tfe atenk


**⭐⭐自注意力机制**

我们在让模型理解句子时 只会让其aggregate给定上下文的prev_words + cur_word信息，未来的words一定会被遮住

1. 考虑最简单的让当前词元和之前词元进行communicate的方式 - 对prev_words和cur_word的 tensor 在**T×C维度** 沿着 **dim="0(T的方向)"** 求和或取平均(√)，索引tensor到这一`t`下，即为模型理解的当前prev_words to cur_word含义

In [48]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn((B, T, C))
x.shape

torch.Size([4, 8, 2])

In [49]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [62]:
# ver1 暴力枚举
xbow1 = torch.zeros_like(x)
for b in range(B):
    for t in range(T):
        x_prev = x[b, :t+1, :]
        xbow1[b, t, :] = torch.mean(x_prev, dim=0)

In [66]:
xbow1

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [67]:
# ver2 可以考虑利用线代的trick实现这一操作 - 初等矩阵（下三角）左乘target tensor可实现 “在dim=t方向上 递进地对每行做前向平均计算”
E = torch.tril(torch.ones(T, T))
E = E / torch.sum(E, dim=1, keepdim=True)
xbow2 = E @ x
xbow2

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [70]:
print(torch.allclose(xbow1, xbow2, atol=1e-6))

True


In [ ]:
# ver3 实际注意力机制使用的计算方式 - softmax

In [72]:
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [73]:
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [74]:
wei = F.softmax(wei, dim = -1)  # 沿着C的维度-最后一维
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [75]:
xbow3 = wei @ x
print(torch.allclose(xbow1, xbow3, atol=1e-6))

True


**ver4 self attention**

核心直觉：我们不希望cur_word对prev_words的信息一视同仁地均匀关注，因为不同token对其他token的关注度应该是具备**差异化**的，而且我们希望这种关注程度能够**动态调整**。

也就是不同token应该以一种依赖数据的方式让prev_words中某些被关注到的信息能够流向该token

每个位置上的token会生成两个向量，也即**query**向量和**key**向量。各自的含义：

**query**：我在查找什么？

**key**：我包含了什么信息？

于是，为了获取某个token和其他token的**亲和度(affinities)**，用该token的`query`**dot product** **其他token的**`key`，点乘结果就是亲和度。

In [88]:
B, T, C = (4, 8, 32)
x = torch.randn(B, T, C)
# 进入到能够捕获注意力的隐空间，使用head_size个注意力头观察
head_size = 16
query = nn.Linear(C, head_size, bias=False)     # "query层 把 x 映射到'我要通过哪些数值去查找'"          ✅ 准确
key = nn.Linear(C, head_size, bias=False)       # "key层 把 x 映射到'别人来查我时我应该暴露什么隐特征'"   ✅ 准确
# (B, T, head_size)
q = query(x)
k = key(x)

# 最终这两套数值在 head_size 维的"以注意力为主题的隐空间"里相遇    ✅ 准确
wei = q @ k.transpose(-2, -1)   # (B, T, head_size) @ (B, head_size, T) -> (B, T, T) - 逐 batch 的 2D 矩阵乘法
wei[0]

tensor([[-1.0261e+00, -3.1015e-01, -5.6500e-01,  2.4903e-01,  6.5753e-01,
          8.5752e-01, -8.4203e-01,  4.0563e-01],
        [ 9.6235e-01,  2.9029e-01, -1.4612e-01, -3.6169e-01, -1.3391e+00,
         -1.1531e+00, -3.6227e-01,  8.6711e-01],
        [-1.5639e+00,  3.1795e-01,  6.9435e-01,  5.3547e-01,  2.2870e+00,
          4.9640e-01,  1.4614e+00, -8.5047e-01],
        [-1.6974e+00, -1.0623e+00, -1.7164e+00,  5.0426e-01,  1.4723e+00,
          2.4977e+00, -6.9100e-01, -7.6791e-01],
        [-8.8560e-01,  1.0240e+00,  1.2833e+00,  1.9657e-01, -8.6979e-01,
         -1.9420e+00,  2.3612e+00, -7.7144e-01],
        [ 6.5730e-01,  1.2725e-01, -8.8164e-01, -7.3746e-01, -3.3705e+00,
         -1.5930e+00,  8.3065e-01, -7.4993e-01],
        [ 8.0284e-01, -1.6284e+00,  1.5984e+00,  2.5482e-01, -6.8990e-01,
          1.5663e+00, -1.1146e+00,  1.9901e-01],
        [-1.0532e-03,  3.0841e-01, -1.3478e+00,  3.6142e-01,  2.2410e+00,
         -3.6309e-01,  6.7285e-01,  1.1045e-01]], grad_fn=<Select

**此时已经得到了每个token关于其他所有token的原始注意力权重方阵（T×T）**，但实际我们希望的是每个token**只对prev_words**产生注意，所以按位置依次抹去那些未来words的注意力值，为了适配后面的softmax操作，在这里的“抹去”表现为硬编码为`-float('inf')`。

In [89]:
tril = torch.tril(wei)
wei = tril.masked_fill(tril == 0, -float('inf'))
wei[0]

tensor([[-1.0261e+00,        -inf,        -inf,        -inf,        -inf,
                -inf,        -inf,        -inf],
        [ 9.6235e-01,  2.9029e-01,        -inf,        -inf,        -inf,
                -inf,        -inf,        -inf],
        [-1.5639e+00,  3.1795e-01,  6.9435e-01,        -inf,        -inf,
                -inf,        -inf,        -inf],
        [-1.6974e+00, -1.0623e+00, -1.7164e+00,  5.0426e-01,        -inf,
                -inf,        -inf,        -inf],
        [-8.8560e-01,  1.0240e+00,  1.2833e+00,  1.9657e-01, -8.6979e-01,
                -inf,        -inf,        -inf],
        [ 6.5730e-01,  1.2725e-01, -8.8164e-01, -7.3746e-01, -3.3705e+00,
         -1.5930e+00,        -inf,        -inf],
        [ 8.0284e-01, -1.6284e+00,  1.5984e+00,  2.5482e-01, -6.8990e-01,
          1.5663e+00, -1.1146e+00,        -inf],
        [-1.0532e-03,  3.0841e-01, -1.3478e+00,  3.6142e-01,  2.2410e+00,
         -3.6309e-01,  6.7285e-01,  1.1045e-01]], grad_fn=<Select

我们希望这些“注意力分数”的数值更好看一点，充分体现**权重**的意义，所以再进行归一化

In [90]:
wei = F.softmax(wei, dim=-1)
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6620, 0.3380, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0584, 0.3832, 0.5584, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0775, 0.1462, 0.0760, 0.7003, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0489, 0.3298, 0.4275, 0.1442, 0.0496, 0.0000, 0.0000, 0.0000],
        [0.4599, 0.2707, 0.0987, 0.1140, 0.0082, 0.0485, 0.0000, 0.0000],
        [0.1563, 0.0137, 0.3463, 0.0903, 0.0351, 0.3353, 0.0230, 0.0000],
        [0.0580, 0.0790, 0.0151, 0.0833, 0.5457, 0.0404, 0.1137, 0.0648]],
       grad_fn=<SelectBackward0>)

-------------
# 你的直觉方向**完全对**，只缺一点关键的几何视角

## 需要补充的关键视角 1：**点积 = 相似度**

为什么 `q @ k.T` 这个操作能表达"匹配度"？

因为**两个向量的点积衡量它们的方向相似程度**：

$$q \cdot k = \|q\| \|k\| \cos\theta$$

- 方向越一致 → 点积越大 → 亲和度越高
- 方向相反 → 点积负很大 → 亲和度极低
- 正交（无关）→ 点积接近 0

所以 query 和 key **被映射到同一个 head_size 维空间是有目的的** —— 只有在同一个空间里，"方向对齐"才有意义。

> 把 head_size 维空间想象成一个"语义检索空间"：
> - 每个 token 通过 `query(x)` 投影出自己的"搜索向量"
> - 每个 token 通过 `key(x)` 投影出自己的"被检索向量"
> - 两两点积 = "我想找的"和"你能提供的"对不对得上

---

## 需要补充的关键视角 2：**Q 和 K 是同一个 x 的两副面孔**

你的代码里 `q` 和 `k` **都来自同一个 x**：

```python
k = key(x)     # 同样的 x
q = query(x)   # 同样的 x
```

这叫 **self-attention**（自注意力）—— 每个 token 同时扮演**查询者**和**被查询者**两个角色。

打个比方：班里每个学生都同时拿着两张卡片：
- 一张写着 "我想找的是…"（query）
- 一张写着 "我能提供的是…"（key）

每两个人之间互相比对（点积），决定 A 听 B 的多少、B 听 A 的多少。**同一个学生用不同的线性层投影出这两张卡片**，所以它们可以表达截然不同的信息——这正是 `key` 和 `query` 是**两个独立的 `nn.Linear`** 的原因（参数不共享）。

如果哪天 K 和 Q 来自**不同**的输入（比如翻译里 Q 来自译文、K 来自原文），就变成 **cross-attention**。

---

## 需要补充的关键视角 3：$Q \cdot V^T$ 

标准写法是：

```python
wei = q @ k.transpose(-2, -1)   # (B, T, hs) @ (B, hs, T) -> (B, T, T)
```

为什么 Q 在左、K 在右？因为这样得到的 `wei[i, j]` 才符合直觉：

> **`wei[i, j]` = token i 的 query 向量 · token j 的 key 向量
> = "token i 在找的东西" 和 "token j 在提供的东西" 的匹配度
> = "token i 想从 token j 那里聚合多少信息" → 想聚合多一点，值就大一点，反之亦然**

回忆我之前讲过的：`wei` 的**行索引**是"接收方"（query），**列索引**是"信息源"（key）。这个角色分工是靠 Q 在左、K 在右建立起来的。

如果你写成 `k @ q.T`，行列含义会颠倒，后面接 causal mask（下三角）的语义就反了——会变成"未来 token 给过去 token 提供 query"，不符合因果生成的设定。

--------------

**Value (V)	真正要返回的内容 —— "如果你要我，这就是我能给的"**

从引入Value层开始，视角要转变为“x现在包含的是 token不暴露出来的、私有的信息”（但`wei`自始至终都表示注意力分数矩阵），而**Value(x)表示为“我被搜索到之后实际给出什么”**，也就是x向外传出去的信息。

具体见`QKV总述.md`

In [93]:
# 完整的 Masked单头self-attetion - DAG上的信息通信机制
B, T, C = (4, 8, 32)
x = torch.randn(B, T, C)
head_size = 16
query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x)
k = key(x)

# 计算注意力分数
wei = q @ k.transpose(-2, -1)   # (B, T, T)
tril = torch.tril(wei)
wei = tril.masked_fill(tril == 0, -float('inf'))
wei = F.softmax(wei, dim=-1)
# 得到由该注意力头产生的结果
v = value(x)
out = wei @ v    # (B, T, head_size)

print(out.shape)
print(out[0])

torch.Size([4, 8, 16])
tensor([[ 1.1724,  0.3628, -0.2099,  0.3909, -0.0054,  0.0763,  0.3279,  0.9612,
         -0.8061, -0.1672, -0.8101,  0.5986, -0.1046, -0.8287,  0.2761,  0.2899],
        [ 0.5802,  0.2325, -0.0148,  0.1957, -0.3053,  0.2404,  0.1652,  0.6263,
         -0.2548, -0.3643,  0.9287,  0.2236, -0.2423, -0.7367,  0.3083, -0.5000],
        [ 0.2260,  0.9517,  0.1464,  0.1731,  0.3091, -0.0781,  0.6182,  0.9483,
          0.3914, -0.8283,  0.5568, -0.0382,  0.0129, -0.9343,  0.3522,  0.1758],
        [ 0.2890,  0.6220,  0.0769,  0.0500,  0.1063,  0.0804,  0.3779,  0.8590,
          0.2612, -0.6294,  0.6567,  0.1178, -0.1560, -0.6555,  0.2633, -0.0504],
        [ 0.2744,  0.3505, -0.0814, -0.1167, -0.0150,  0.2067,  0.1529,  0.7188,
          0.1640, -0.3952,  0.4394,  0.3223, -0.2659, -0.2470,  0.1250, -0.0198],
        [ 0.2554,  0.3839,  0.0310,  0.1096, -0.1446, -0.0029,  0.2914,  0.6521,
          0.1579, -0.4940,  0.8979,  0.1214, -0.1498, -0.5884,  0.3353, -0.3287],

为了降低原始attention score矩阵的方差，从而削减softmax对输入大小的敏感性，初始阶段就将$q \cdot k^T$的结果进行$/ \sqrt d_k$(`head_size`)，直接**减小了标准差**，从而控制softmax的输入在一个“温和”的区间。

In [ ]:
wei = q @ k.transpose(-2, -1) * head_size**-0.5

**从单头注意力到多头注意力**

注意力机制可以灵活适配为单头/多头的

如果是单头的，

结构约定：`head_size == n_embd`

思路：寄希望于在一个完整维度的隐空间里表达出所有需要捕获的关系

问题：一个注意力头只能学到一种关注模式——要么句法、要么语义、要么远距离依赖，难以兼顾。表达瓶颈明显 ❌


而通常使用的是多头，

结构约定：`num_heads × head_size == n_embd`
例如 n_embd = 384, num_heads = 6, head_size = 64

思路：把 embedding 空间切成 num_heads 个低维子空间，每个 head 在自己的子空间里独立做一遍 attention，最后沿最后一维拼接 (concat) 回 n_embd

优势：
> 分工：不同 head 自然学到不同关注模式（句法 / 语义 / 邻近 / 远距离…）
>
> 并行：多个 head 同时计算，几乎不增加 wall-clock 时间
>
> 总参数量基本不变：因为单头变窄了 (head_size 从 n_embd 缩到 n_embd/num_heads)



多头注意力最后还需要一个投影层（`Linear`）

proj 不是为了"形状对齐"—— 而是为了让各个头融合通信

`torch.cat(h_outs, dim=-1)` 已经把形状从 `num_heads × (B, T, head_size)` 拼回 `(B, T, C)` 了。但形状对了不等于信息整合好了。proj 解决的是"信息融合"问题，不是"形状对齐"问题。

配置：

```python
batch_size = 32
block_size = 8
n_embd = 32
num_heads = 4
head_size = n_embd // num_heads
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
```

在仅1个block下，各模块添加之后的`loss`变化：

base: token_embedding + position_embedding
> step 4500: train loss 2.4926, val loss 2.5021

(1)add: Masked Multi-Head Attention
> step 4500: train loss 2.2574, val loss 2.2731

(2)add: FeedForwad
> step 4500: train loss 2.2075, val loss 2.2345

(3)add: Residual Connection
> step 4500: train loss 2.1420, val loss 2.1847

(4)add: Pre-LayerNorm
> step 4500: train loss 2.1152, val loss 2.1718

(5)add: Dropout in FC-layer's position
> step 4500: train loss 2.1883, val loss 2.2190 （↓）

Block completed！

